In [8]:
from ddsp import DDSP, AudioDataset
from ddsp.callbacks import BetaWarmupEpochCallback
from ddsp.synths import NoiseBandSynth, SineSynth
from ddsp.utils import find_checkpoint

from lightning.pytorch.callbacks import ModelCheckpoint
from lightning import Trainer

from IPython.display import Audio, display

from torch.utils.data import DataLoader, Subset, random_split

import torch
import umap

torch.set_float32_matmul_precision('medium')
torch.set_default_tensor_type('torch.cuda.FloatTensor')
torch.set_default_device('cuda')


import os

experiment_root = '/home/btadeusz/code/ddsp_vae/experiments/smooth_mss_loss/'
training_root = os.path.join(experiment_root, 'lightning_logs')

In [2]:
# Dataset parameters
chunk_duration = 2.0
sampling_rate = 44100
n_signal = chunk_duration * sampling_rate
batch_size = 8

# Model parameters
latent_size = num_params = 4
max_freq = 22050
n_sines = 100

# Training parameters
warmup_start = 300
warmup_end = 500
beta = 1.0
max_epochs = 1000
learning_rate = 1e-4


In [4]:
def get_dataset_split(dataset_path, validation_split=0.2):
  """
  Splits the dataset into training and validation sets.
  """
  generator = torch.Generator(device='cuda')

  dataset_A = AudioDataset(dataset_path=dataset_path, n_signal=n_signal)
  total_len = len(dataset_A)

  val_len = int(validation_split * total_len)  # 20% for validation
  indices = torch.randperm(total_len, generator=generator)

  val_indices = indices[:val_len]
  train_indices = indices[val_len:]

  train_set = Subset(dataset_A, train_indices)
  val_set = Subset(dataset_A, val_indices)

  train_loader = DataLoader(train_set, batch_size, shuffle=True, num_workers=0, generator=generator)
  val_loader = DataLoader(val_set, batch_size, shuffle=False, num_workers=0, generator=generator)

  return train_loader, val_loader

def examine_model(model, val_loader):
  """
  Examines the trained model by displaying original and autoencoded audio samples,
  as well as reporting on the loss.
  """
  model = model.cuda()
  model.eval()
  with torch.no_grad():
    x_audio = next(iter(val_loader))
    x_audio = x_audio.to('cuda')

    # Forward pass through the model
    y_audio = model(x_audio).squeeze(1)

    print(x_audio.shape, y_audio.shape)

    # Display original and reconstructed audio
    for j in range(x_audio.shape[0]):
      print(f"Sample {j + 1}:")
      x_audio_j = x_audio[j].cpu()
      y_audio_j = y_audio[j].cpu()

      display(Audio(x_audio_j.numpy(), rate=sampling_rate))
      display(Audio(y_audio_j.numpy(), rate=sampling_rate))


In [ ]:
sines = SineSynth.to_config(fs=sampling_rate, n_sines=n_sines)

ddsp = DDSP(
  synth_configs=[sines],
  fs=sampling_rate,
  latent_size=latent_size,
  num_params=num_params,
  learning_rate=learning_rate,
  perceptual_loss_weight=0,
  plateau_patience=200,
).to('cuda')

best_checkpoint_callback = ModelCheckpoint(
  filename='best',
  monitor='val_loss',
  mode='min',
  save_top_k=1,
  save_last=True,
  dirpath=training_root,
)

trainer = Trainer(
  callbacks = [best_checkpoint_callback],
  max_epochs=1000,
  accelerator='cuda',
  precision=16
)

ckpt = os.path.join(training_root, 'version_5', 'checkpoints', 'epoch=249-step=16250.ckpt')

train_loader, val_loader = get_dataset_split('/mnt/mariadata/datasets/syntex/am_and_chirps/DS_BasicAM_1.1/audio')
trainer.fit(
    model=ddsp,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
    ckpt_path=ckpt
)

/home/btadeusz/miniconda3/envs/ddsp/lib/python3.11/site-packages/torchaudio/functional/functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(
/home/btadeusz/miniconda3/envs/ddsp/lib/python3.11/site-packages/lightning/fabric/connector.py:571: `precision=16` is supported for historical reasons but its usage is discouraged. Please set your precision to 16-mixed instead!
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Restoring states from the checkpoint path at /home/btadeusz/code/ddsp_vae/experiments/smooth_mss_loss/lightning_logs/version_5/checkpoints/epoch=249-step=16250.ckpt
/home/btadeusz/miniconda3/envs/ddsp/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:360: The dirpath has changed from '/home/btad

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/btadeusz/miniconda3/envs/ddsp/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.
/home/btadeusz/miniconda3/envs/ddsp/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=27` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [7]:
examine_model(ddsp, val_loader)

torch.Size([8, 88200]) torch.Size([8, 88192])
Sample 1:


Sample 2:


Sample 3:


Sample 4:


Sample 5:


Sample 6:


Sample 7:


Sample 8:
